# Local Multi-Agent Research Assistant — Walkthrough

This notebook runs the assistant **on a real local model via Ollama** (no paid APIs) and shows:

1. the task and the example documents,
2. the workflow,
3. a full run on a representative question (trace, evidence, citations, report),
4. evaluation results and the multi-agent vs. single-agent comparison, and
5. a discussed failure case.

> **Not mocked.** Every run below calls the local model. Execute top-to-bottom from a clean kernel;
> outputs are produced live. Requires Ollama running with the configured models
> (`research-assistant doctor`). Set `RA_CHAT_MODEL` to pick the model (default `qwen3.5:9b`;
> `qwen2.5:7b` is a faster fallback).

In [1]:
import sys
from pathlib import Path

# Make the package importable whether run from repo root or notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from research_assistant.config import Settings
from research_assistant.models import check_ollama

settings = Settings.from_env()
status = check_ollama(settings)
print("Ollama reachable:", status.reachable)
print("Chat model:", settings.chat_model, "| available:", status.has(settings.chat_model))
print("Embedding model:", settings.embedding_model, "| available:", status.has(settings.embedding_model))
assert status.reachable, "Start Ollama with `ollama serve` and pull the models (see README)."

Ollama reachable: True
Chat model: qwen3.5:9b | available: True
Embedding model: qwen3-embedding:0.6b | available: True


## 1. The task and the documents

The assistant answers research questions about **AI governance** over a small, licensed corpus: the
**EU AI Act** (Regulation (EU) 2024/1689) and two **NIST** AI Risk Management Framework documents.
Each document is parsed into citable passages — the EU AI Act by Article/Recital/Annex, the NIST
PDFs by page — so every quoted fact keeps an identifiable location.

In [2]:
import json

from research_assistant.corpus import load_passages, local_sources

manifest = json.loads((settings.corpus_dir / "manifest.json").read_text())
for src in manifest["sources"]:
    print(f"- {src['short_title']}: {src['title'][:70]}…")
    print(f"    published {src['published']} · {src['license'][:60]}…")

passages = load_passages(settings.corpus_dir)
sources = local_sources(settings.corpus_dir)
print(f"\nTotal passages: {len(passages)} across {len(sources)} documents")
# Show one real passage with its citable location.
art53 = next(p for p in passages if p.id.startswith("eu-ai-act:art-53"))
print(f"\nExample passage [{art53.id}] — {art53.location}:\n{art53.text[:400]}…")

- EU AI Act: Regulation (EU) 2024/1689 laying down harmonised rules on artificial i…
    published 2024-07-12 · EUR-Lex legal notice: legal documents published in EUR-Lex m…
- NIST AI RMF 1.0: Artificial Intelligence Risk Management Framework (AI RMF 1.0), NIST A…
    published 2023-01-26 · Work of the U.S. Government. NIST states that information on…
- NIST GenAI Profile: Artificial Intelligence Risk Management Framework: Generative Artifici…
    published 2024-07-26 · Work of the U.S. Government. NIST states that information on…

Total passages: 762 across 3 documents

Example passage [eu-ai-act:art-53:1] — Article 53 (Obligations for providers of general-purpose AI models):
1. Providers of general-purpose AI models shall:
(a) draw up and keep up-to-date the technical documentation of the model, including its training and testing process and the results of its evaluation, which shall contain, at a minimum, the information set out in Annex XI for the purpose of providing it, upon requ

## 2. The workflow

A **supervisor** picks the next step (delegate to a web researcher, delegate to a document analyst,
or finish). Two workers do the research; a **writer** composes a cited report and a deterministic
**finalize** step validates every citation. The model only chooses *what next* and *tool arguments*;
routing, budgets, guards, evidence quoting and citation validation are deterministic code.

In [3]:
from research_assistant.graph import build_multi_agent_graph

graph = build_multi_agent_graph()
try:
    print(graph.get_graph().draw_ascii())
except Exception:
    print("Nodes:", list(graph.get_graph().nodes))

Nodes: ['__start__', 'supervisor', 'web_researcher', 'document_analyst', 'writer', 'finalize', '__end__']


## 3. A full run (offline, multi-agent)

We run a question answerable entirely from the local corpus, with web search disabled for
reproducibility. Building the index embeds the corpus once (then it is cached on disk).

In [4]:
import time

from research_assistant.engine import ResearchAssistant

assistant = ResearchAssistant(settings.with_overrides(web_enabled=False))
print("Index:", assistant.index_summary())

question = (
    "What transparency and documentation obligations does the EU AI Act place on "
    "providers of general-purpose AI models?"
)
t0 = time.time()
out = assistant.run(question, mode="multi_agent")
print(f"Completed in {time.time() - t0:.0f}s · status={out.report.run.status}")

Index: 762 passages from 3 documents; hybrid (BM25 + embeddings)


Completed in 69s · status=completed


### Execution trace
Every decision, tool call and retrieved source is recorded.

In [5]:
import pandas as pd

trace_df = pd.DataFrame(
    {
        "t": round(e.t, 1),
        "agent": e.agent,
        "kind": e.kind,
        "name": e.name,
        "detail": str(e.detail.get("summary") or e.detail.get("args") or "")[:90],
    }
    for e in out.events
)
trace_df

,t,agent,kind,name,detail
0,0.0,multi_agent,node,run_start,What transparency and documentation obligation...
1,3.5,supervisor,llm,llm_call,
2,3.5,supervisor,decision,document_analysis,"Search the local documents, specifically eu-ai..."
3,3.5,document_analyst,node,start,"Search the local documents, specifically eu-ai..."
4,6.6,document_analyst,llm,llm_call,
5,6.6,document_analyst,tool_call,search_documents,"{""query"": ""general-purpose AI models transpare..."
6,7.1,document_analyst,tool_result,search_documents,5 passages
7,33.6,document_analyst,llm,llm_call,
8,33.6,document_analyst,tool_call,record_evidence,"{""passage_id"": ""[eu-ai-act:rct-101:1]"", ""note""..."
9,33.6,document_analyst,tool_result,record_evidence,E1 ← eu-ai-act:rct-101:1


### Evidence, citations and the report
Evidence quotes are copied verbatim from the corpus; the analyst only writes the note.

In [6]:
for e in out.report.evidence:
    print(f"[{e.id}] {e.source_id} — {e.location}")
    print(f"     note: {e.note}")
    print(f"     quote: {e.quote[:160]}…\n")

c = out.report.citation_check
print(f"Citation validity: {c.valid_citations}/{c.total_citations} resolve to recorded evidence "
      f"({c.validity_rate:.0%}); {c.uncited_findings} uncited findings.")

[E1] eu-ai-act — Recital 101
     note: Recital 101 states that providers of general-purpose AI models must prepare and keep up-to-date technical documentation for the purpose of making it available to the AI Office and national competent authorities.
     quote: (101) Providers of general-purpose AI models have a particular role and responsibility along the AI value chain, as the models they provide may form the basis f…

[E2] eu-ai-act — Article 53 (Obligations for providers of general-purpose AI models)
     note: Article 53(1) requires providers of general-purpose AI models to draw up and keep up-to-date technical documentation including training and testing processes, evaluation results, and information from Annex XI for the AI Office and national competent authorities.
     quote: 1. Providers of general-purpose AI models shall:
(a) draw up and keep up-to-date the technical documentation of the model, including its training and testing pr…

[E3] eu-ai-act — Article 53 (Obligation

In [7]:
from IPython.display import Markdown

from research_assistant.report import report_to_markdown

Markdown(report_to_markdown(out.report))

# Research report: What transparency and documentation obligations does the EU AI Act place on providers of general-purpose AI models?

*Mode: multi_agent · Model: qwen3.5:9b · Web: off · Status: completed · 69s · 6 LLM calls · 6 tool calls*

## Answer

Providers of general-purpose AI models must prepare and keep up-to-date technical documentation for the purpose of making it available, upon request, to the AI Office and national competent authorities [E1]. This technical documentation must include information on the model's training and testing processes, evaluation results, and the elements set out in Annex XI [E2]. Additionally, providers must draw up and make publicly available a sufficiently detailed summary about the content used for training of the general-purpose AI model, according to a template provided by the AI Office [E3]. However, these obligations do not apply to providers releasing models under free and open-source licenses where parameters are made publicly available, unless the model has systemic risks [E3].

## Key findings

- Providers of general-purpose AI models must prepare and keep up-to-date technical documentation for the purpose of making it available, upon request, to the AI Office and national competent authorities. [E1]
- Technical documentation must include information on the model's training and testing processes, evaluation results, and the elements set out in Annex XI. [E2]
- Providers must draw up and make publicly available a sufficiently detailed summary about the content used for training of the general-purpose AI model, according to a template provided by the AI Office. [E3]
- These obligations do not apply to providers releasing models under free and open-source licenses where parameters are made publicly available, unless the model has systemic risks. [E3]

## Limitations and open questions

- The specific elements required in Annex XI and Annex XII are not detailed in the provided evidence text.
- Web search was disabled for this run; only the local document corpus was consulted.
- All cited evidence comes from the local corpus, which reflects the document versions listed under Sources (e.g. the EU AI Act as published in July 2024). Later amendments, guidance or implementation changes are not covered.

## Evidence

**[E1]** eu-ai-act — Recital 101  
*Note (document_analyst):* Recital 101 states that providers of general-purpose AI models must prepare and keep up-to-date technical documentation for the purpose of making it available to the AI Office and national competent authorities.  
> (101) Providers of general-purpose AI models have a particular role and responsibility along the AI value chain, as the models they provide may form the basis for a range of downstream systems, often provided by downstream providers that necessitate a good understanding of the models and their capabilities, both to enable the integration of such models into their products, and to fulfil their obligations under this or other regulations. Therefore, proportionate transparency measures should be laid down, including the drawing up and keeping up to date of documentation, and the provision of …

**[E2]** eu-ai-act — Article 53 (Obligations for providers of general-purpose AI models)  
*Note (document_analyst):* Article 53(1) requires providers of general-purpose AI models to draw up and keep up-to-date technical documentation including training and testing processes, evaluation results, and information from Annex XI for the AI Office and national competent authorities.  
> 1. Providers of general-purpose AI models shall:
(a) draw up and keep up-to-date the technical documentation of the model, including its training and testing process and the results of its evaluation, which shall contain, at a minimum, the information set out in Annex XI for the purpose of providing it, upon request, to the AI Office and the national competent authorities;
(b) draw up, keep up-to-date and make available information and documentation to providers of AI systems who intend to integrate the general-purpose AI model into their AI systems. Without prejudice to the need to observe …

**[E3]** eu-ai-act — Article 53 (Obligations for providers of general-purpose AI models)  
*Note (document_analyst):* Article 53(2) requires providers of general-purpose AI models to draw up and make publicly available a sufficiently detailed summary about the content used for training, according to a template provided by the AI Office.  
> (d) draw up and make publicly available a sufficiently detailed summary about the content used for training of the general-purpose AI model, according to a template provided by the AI Office.
2. The obligations set out in paragraph 1, points (a) and (b), shall not apply to providers of AI models that are released under a free and open-source licence that allows for the access, usage, modification, and distribution of the model, and whose parameters, including the weights, the information on the model architecture, and the information on model usage, are made publicly available. This exception …

## Sources

- **eu-ai-act** — Regulation (EU) 2024/1689 laying down harmonised rules on artificial intelligence (Artificial Intelligence Act) <https://eur-lex.europa.eu/eli/reg/2024/1689/oj>  
  published 2024-07-12; license: EUR-Lex legal notice: legal documents published in EUR-Lex may be re-used for commercial or non-commercial purposes …

## Citation check (automated)

- Citation validity: 7/7 references resolve to recorded evidence (100%).
- Findings without a valid citation: 0.
- Answer contains inline citations: yes.

> Validity means a cited ID maps to evidence recorded this run. It does **not** verify that the evidence supports the claim — see the evaluation for sampled support checks.


## 4. Evaluation

A small **grounded** evaluation set pairs each question with expected sources and findings (verified
by hand against the corpus). We evaluate both the **multi-agent** workflow and the **single-agent**
baseline on the offline questions.

**Honest labelling:** the checks below are *automated and deterministic* — citation validity,
expected-source recall, expected-finding coverage (keyword match), and a *lexical grounding proxy*
(word overlap between a claim and its cited evidence; a weak signal, not entailment). Whether a
claim is *truly supported* is left to a human review step (shown at the end). The model scoring its
own output is not treated as validation.

The full 3-question × 2-mode sweep takes several minutes, so it is run once via
`scripts/run_evaluation.py` and the results are committed to `data/eval/results-latest.json`.
We **load** those results here for the aggregate comparison, then run one **live** single-agent
pass below so this notebook still exercises the baseline on the real model.

In [8]:
eval_results = json.loads((settings.data_dir / "eval" / "results-latest.json").read_text())
print(f"Loaded evaluation: model={eval_results['model']}, web={eval_results['web_enabled']}")
summary = pd.DataFrame(eval_results["per_mode_aggregate"]).T
summary[["completion_rate", "mean_citation_validity", "mean_finding_coverage",
         "mean_grounding_proxy", "mean_duration_s", "mean_llm_calls"]]

Loaded evaluation: model=qwen3.5:9b, web=False


,completion_rate,mean_citation_validity,mean_finding_coverage,mean_grounding_proxy,mean_duration_s,mean_llm_calls
multi_agent,1.000,1.000,0.667,0.771,200.633,16.667
single_agent,0.667,0.667,0.417,0.493,186.200,12.333


**Reading the comparison.** The multi-agent workflow completed and validly cited *all* questions;
the single-agent baseline (same tools, same empty-evidence safeguard) completed fewer and covered
less on average. But it is not uniformly worse — on the EU-vs-NIST comparison question the single
agent actually covered more — so the win is **reliability and source breadth**, at the cost of more
LLM calls. Small-n on a local 9B model: directional, not a benchmark.

### Live single-agent run (same question as Section 3)
One agent with the same tools and budget. Compare its evidence and citations to the multi-agent run.

In [9]:
single = assistant.run(question, mode="single_agent")
sr = single.report
print(f"single-agent: status={sr.run.status} · {sr.run.duration_s:.0f}s · {sr.run.llm_calls} LLM calls · "
      f"{len(sr.evidence)} evidence · citations {sr.citation_check.valid_citations}/"
      f"{sr.citation_check.total_citations} valid")

single-agent: status=partial · 41s · 3 LLM calls · 4 evidence · citations 4/4 valid


### Manual claim-support review (the honest check)
Automated validity only proves a citation *resolves*. Whether evidence *supports* a claim is for a
human to judge — `scripts/run_evaluation.py --manual` writes one row per finding to a CSV with the
`supported` column left blank. Here are the rows for the multi-agent showcase run, with the lexical
grounding proxy as a hint (not a verdict).

In [10]:
from research_assistant.evaluation import manual_review_rows

review_df = pd.DataFrame(
    {
        "finding": row.finding[:70],
        "evidence": "; ".join(row.evidence_ids),
        "grounding_proxy": row.grounding_proxy,
        "supported (human fills in)": row.supported or "?",
    }
    for row in manual_review_rows(out.report, "gpai-obligations")
)
review_df

,finding,evidence,grounding_proxy,supported (human fills in)
0,Providers of general-purpose AI models must pr...,E1,0.250,?
1,Technical documentation must include informati...,E2,0.688,?
2,Providers must draw up and make publicly avail...,E3,0.955,?
3,These obligations do not apply to providers re...,E3,0.421,?


## 5. A failure case

Early runs surfaced a real failure with `qwen3.5:9b`: the document analyst **searched but never
recorded evidence**, and the supervisor then "finished" — producing an empty, failed report.

**Fixes applied** (both in the repo):
1. the analyst prompt now states that searching alone is not enough and it must call
   `record_evidence` before finishing; and
2. a supervisor **guard** forces re-analysis (up to a retry budget) when `finish` is chosen with
   zero recorded evidence, instead of emitting a hollow report.

The lesson: when a local model misbehaves, back it with a deterministic guard rather than trusting a
prompt tweak. We can see the guard in action if a run ever records nothing — the run status becomes
`failed` only after the retry budget is spent, and the limitations section says so explicitly.

In [11]:
# The run's own self-report of limitations and any warnings (deterministic, not model-written):
for lim in out.report.limitations:
    print("-", lim)
if out.report.run.errors:
    print("\nWarnings:")
    for err in out.report.run.errors:
        print("-", err[:120])

- The specific elements required in Annex XI and Annex XII are not detailed in the provided evidence text.
- Web search was disabled for this run; only the local document corpus was consulted.
- All cited evidence comes from the local corpus, which reflects the document versions listed under Sources (e.g. the EU AI Act as published in July 2024). Later amendments, guidance or implementation changes are not covered.


---
**Takeaway.** The assistant produces a structured, source-backed report with an auditable trace,
preserves source identifiers end to end, copies evidence verbatim, and validates citations in code.
The multi-agent design earns its keep on multi-part questions; a single agent or plain RAG is
preferable for simple lookups — quantified by the baseline above.